In [2]:
import os
import csv
from pathlib import Path
from collections import Counter

# ============================================================
# CONFIG
# ============================================================
DATASET_ROOT = Path("/kaggle/input/datasets/fahadhasan93/eye-diseases-classification/dataset")
OUTPUT_CSV   = Path("/kaggle/working/prompt_meta.csv")   # writable location on Kaggle

# Consistent prompts for each class (edit if you want different wording)
CLASS_PROMPTS = {
    "cataract": (
        "high quality fundus photograph of an eye with cataract, "
        "cloudy lens, reduced clarity, medical ophthalmology image, "
        "detailed retinal view, clinical photography"
    ),
    "diabetic_retinopathy": (
        "high quality fundus photograph of an eye with diabetic retinopathy, "
        "microaneurysms, hemorrhages, exudates, neovascularization, "
        "medical ophthalmology image, detailed retinal view, clinical photography"
    ),
    "glaucoma": (
        "high quality fundus photograph of an eye with glaucoma, "
        "optic disc cupping, increased cup-to-disc ratio, "
        "medical ophthalmology image, detailed retinal view, clinical photography"
    ),
    "normal": (
        "high quality fundus photograph of a healthy normal eye, "
        "clear optic disc, healthy retinal vessels, no pathology, "
        "medical ophthalmology image, detailed retinal view, clinical photography"
    ),
}

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

# ============================================================
# MAIN
# ============================================================
def main():
    rows = []

    if not DATASET_ROOT.exists():
        raise FileNotFoundError(f"Dataset folder not found: {DATASET_ROOT}")

    for class_dir in sorted(DATASET_ROOT.iterdir()):
        if not class_dir.is_dir():
            continue

        class_name = class_dir.name
        prompt = CLASS_PROMPTS.get(class_name)

        if prompt is None:
            print(f"[WARNING] No prompt defined for class '{class_name}' – skipping")
            continue

        for img_path in sorted(class_dir.rglob("*")):
            if img_path.suffix.lower() in IMAGE_EXTS and img_path.is_file():
                rows.append({
                    "Image name": img_path.name,
                    "Class": class_name,
                    "prompt_tr": prompt,
                })

    # Write CSV
    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["Image name", "Class", "prompt_tr"])
        writer.writeheader()
        writer.writerows(rows)

    print(f"✅ Done! Wrote {len(rows)} rows to {OUTPUT_CSV}")
    print("\nClass distribution:")
    counts = Counter(r["Class"] for r in rows)
    for cls, cnt in sorted(counts.items()):
        print(f"  {cls:25s} : {cnt}")

if __name__ == "__main__":
    main()

✅ Done! Wrote 4217 rows to /kaggle/working/prompt_meta.csv

Class distribution:
  cataract                  : 1038
  diabetic_retinopathy      : 1098
  glaucoma                  : 1007
  normal                    : 1074
